# Chapter 13.2. 로봇 환경 탐색 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter13_2_robot_envs.ipynb)

책 본문: [Chapter 13.2](https://smhanlab.com/book-ml/kor/ml2/chapter13/2.html)

이 노트북은 책 13.2절의 실험을 그대로 실행합니다:
1. **평가 프로토콜** — 창(50 vs 200스텝)과 시드가 비교에 미치는 영향 (Pendulum-v1)
2. **PD 그리드 탐색** — $k_p \in \{1,2,4,8\}$, $k_d \in \{0, 0.5, 1, 2, 4\}$ 20개 정책
3. **점수 vs 궤적** — 최선 정책이 진자를 *세워 유지*하는지 상태 궤적으로 확인
4. **Reacher/Ant** — 무작위 정책 평가 + Ant의 4항 보상 장부 재구성 + 스텝당 시간 (MuJoCo 필요)

## 1. 평가 프로토콜: 창(horizon)과 시드

`Pendulum-v1`은 classic_control 계열이라 **MuJoCo 설치 없이** 돌아가므로, 여기부터 5번 셀까지는 어떤 환경에서든 재현됩니다. 첫 질문: 같은 코드로 50스텝을 돌리면 무작위 정책과 PD 정책(\(k_p = 2, k_d = 0.5\), 책의 첫 코드와 동일)을 구분할 수 있는가?

In [1]:
import matplotlib
matplotlib.use("Agg")
from matplotlib import font_manager
import matplotlib.pyplot as plt
kr = [f.name for f in font_manager.fontManager.ttflist if "Noto Sans CJK KR" in f.name]
if kr: plt.rcParams["font.sans-serif"] = [kr[0]]
plt.rcParams["axes.unicode_minus"] = False

import numpy as np
import gymnasium as gym

IMG = "/home/smhan/book-ml/kor/src/images"
DT = 0.05  # 1스텝 = 0.05초 (50스텝 = 2.5초, 200스텝 = 10초)

def eval_pendulum(policy, n_steps, seeds=(0, 1, 2, 3)):
    """policy(theta, thdot) -> torque. 시드별 누적 보상을 반환."""
    env = gym.make("Pendulum-v1")
    vals = []
    for seed in seeds:
        s, _ = env.reset(seed=seed)
        tot = 0.0
        for _ in range(n_steps):
            th = np.arctan2(s[1], s[0])
            a = np.array([np.clip(policy(th, s[2]), -2.0, 2.0)])
            s, r, term, trunc, _ = env.step(a)
            tot += r
            if term or trunc:
                break
        vals.append(tot)
    env.close()
    return vals

def eval_random(n_steps, seeds=(0, 1, 2, 3)):
    """무작위 정책 — 에피소드마다 시드된 RNG로 행동을 뽑아
    (전역 numpy RNG 상태를 무관하게) 어떤 순서로 실행해도 결과가 같다."""
    env = gym.make("Pendulum-v1")
    lo, hi = env.action_space.low, env.action_space.high
    vals = []
    for seed in seeds:
        rng = np.random.default_rng(1000 + seed)
        s, _ = env.reset(seed=seed)
        tot = 0.0
        for _ in range(n_steps):
            a = rng.uniform(lo, hi)
            s, r, term, trunc, _ = env.step(a)
            tot += r
            if term or trunc:
                break
        vals.append(tot)
    env.close()
    return vals

print("50스텝 창 (2.5초), 시드 0~3 — 무작위 vs PD(kp=2, kd=0.5):")
for seed in range(4):
    r = eval_random(50, (seed,))[0]
    p = eval_pendulum(lambda th, thd: -2.0 * th - 0.5 * thd, 50, (seed,))[0]
    print(f"  seed{seed}: random {r:8.1f} | PD(2, 0.5) {p:8.1f}")

50스텝 창 (2.5초), 시드 0~3 — 무작위 vs PD(kp=2, kd=0.5):
  seed0: random   -246.7 | PD(2, 0.5)   -238.3
  seed1: random   -234.2 | PD(2, 0.5)   -156.8
  seed2: random   -300.5 | PD(2, 0.5)   -277.2
  seed3: random   -403.1 | PD(2, 0.5)   -381.9


50스텝 창에서는 두 정책의 숫자가 겹쳐 "어느 쪽이 좋은지" 판정하기 어렵다 — **창이 너무 짧으면 초기 이동(첫 휘청임)이 점수를 지배**하고, 시작 자세가 랜덤이라 운의 비중이 크다.

## 2. 창을 맞췄을 때: 200스텝(10초)에서 PD vs 무작위가 갈라진다

In [2]:
r200 = eval_random(200, (0, 1, 2))
p200 = eval_pendulum(lambda th, thd: -8.0 * th - 2.0 * thd, 200, (0, 1, 2))
print("200스텝 창 (10초), 시드 0~2:")
print("  random:", [round(v, 1) for v in r200], " 평균", round(float(np.mean(r200)), 1))
print("  PD(8,2):", [round(v, 1) for v in p200], " 평균", round(float(np.mean(p200)), 1))
print(f"\n  같은 창·같은 시드에서 비교하면: PD(8,2)가 무작위 대비 누적 벌을 "
      f"{float(np.mean(r200)) / float(np.mean(p200)):.1f}배 줄인다")

200스텝 창 (10초), 시드 0~2:
  random: [np.float64(-914.1), np.float64(-884.4), np.float64(-1150.6)]  평균 -983.1
  PD(8,2): [np.float64(-274.8), np.float64(-0.5), np.float64(-1089.3)]  평균 -454.9

  같은 창·같은 시드에서 비교하면: PD(8,2)가 무작위 대비 누적 벌을 2.2배 줄인다


## 3. PD 그리드 탐색: 20개 정책의 200스텝 점수 (각 3시드 평균)

사람이 직접 설계한 정책은 **파라미터가 두 개(\(k_p, k_d\))**라서 그리드 탐색이 가능하다 — 그런데 이것이 고차원 로봇(13.3절의 105→8차원)에서는 죽는 방식이기도 하다.

In [3]:
KPS = [1, 2, 4, 8]
KDS = [0, 0.5, 1, 2, 4]
grid = {}
for kp in KPS:
    for kd in KDS:
        grid[(kp, kd)] = float(np.mean(eval_pendulum(lambda th, thd, kp=kp, kd=kd: -kp * th - kd * thd, 200, (0, 1, 2))))
print("200스텝 × 3시드 평균 누적 보상 (책의 표와 동일):\n")
print("        " + "  ".join(f"{kd:<9}" for kd in KDS))
for kp in KPS:
    print(f"kp={kp:<2}: " + "  ".join(f"{grid[(kp, kd)]:9.1f}" for kd in KDS))
best = max(grid, key=grid.get)
print(f"\n  최선: (kp, kd) = {best}  →  {grid[best]:.1f}")

200스텝 × 3시드 평균 누적 보상 (책의 표와 동일):

        0          0.5        1          2          4        
kp=1 :    -889.7    -1365.8    -1410.1    -1401.8    -1374.2
kp=2 :    -947.3     -995.4    -1150.8    -1333.1    -1336.2
kp=4 :    -767.9     -785.8     -821.7    -1001.3    -1094.0
kp=8 :    -553.3     -455.7     -455.0     -454.9     -697.0

  최선: (kp, kd) = (8, 2)  →  -454.9


## 4. 그리드 히트맵 (그림 1)

밝을수록(0에 가까울수록) 좋다. \(k_p = 8\) 행이 가장 밝고, \(k_d = 4\) 열은 \(k_p\)가 클수록 오히려 어두워진다(과감쇠). 빨간 점선은 같은 프로토콜의 무작위 정책 평균이다.

In [4]:
fig, ax = plt.subplots(figsize=(8.5, 4.6))
M = np.array([[grid[(kp, kd)] for kd in KDS] for kp in KPS])
im = ax.imshow(M, cmap="RdYlGn", aspect="auto")
for i in range(len(KPS)):
    for j in range(len(KDS)):
        v = M[i, j]
        ax.text(j, i, f"{v:.0f}", ha="center", va="center",
                color="white" if v < -950 else "black", fontsize=11)
ax.set_xticks(range(len(KDS)), [str(k) for k in KDS])
ax.set_yticks(range(len(KPS)), [str(k) for k in KPS])
ax.set_xlabel("$k_d$ (미분 이득)")
ax.set_ylabel("$k_p$ (비례 이득)")
ax.set_title("PD 그리드 — 200스텝 누적 보상 (3시드 평균) [밝을수록 좋음]")
rand_mean = float(np.mean(eval_random(200, (0, 1, 2))))
ax.contour(M, levels=[rand_mean], colors="red", linestyles="dashed", linewidths=1.5)
ax.text(4.45, 0.15, f"무작위 기준 \u2248 {rand_mean:.0f}", color="red",
        fontsize=10, ha="right", va="bottom")
cb = fig.colorbar(im, ax=ax, shrink=0.9)
cb.set_label("200스텝 누적 보상 (음수 = 벌, 0에 가까울수록 좋음)", fontsize=9)
fig.savefig(IMG + "/ch13_2_pd_grid.svg", bbox_inches="tight")
plt.show()

## 5. 점수 vs 궤적: 최선 정책은 진자를 *세워 유지*하는가? (그림 2)

누적 보상은 "위쪽을 세운 채 유지한다"는 진짜 과제의 **프록시**다. 같은 시작(시드 0)에서 그리드의 최선 $(k_p, k_d) = (8, 2)$와 $(4, 1)$의 궤적을 10초(200스텝) 그려서, 13.1절이 수학으로 말한 $k_p > 5$ 조건을 실험으로 확인한다.

In [5]:
def trajectory(kp, kd, steps=200, seed=0):
    env = gym.make("Pendulum-v1")
    s, _ = env.reset(seed=seed)
    th = [np.arctan2(s[1], s[0])]; td = [s[2]]
    for _ in range(steps):
        theta = np.arctan2(s[1], s[0])
        a = np.array([np.clip(-kp * theta - kd * s[2], -2.0, 2.0)])
        s, r, term, trunc, _ = env.step(a)
        th.append(np.arctan2(s[1], s[0])); td.append(s[2])
    env.close()
    return np.array(th), np.array(td), np.arange(steps + 1) * DT

th1, td1, tt = trajectory(8, 2)   # 그리드의 최선
th2, td2, _ = trajectory(4, 1)    # k_p < 5 -- 수렴하지 않음

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].plot(tt, th1, color="green", label="PD ($k_p$=8, $k_d$=2)  <- 최선")
axes[0].plot(tt, th2, color="orange", label="PD ($k_p$=4, $k_d$=1)")
axes[0].axhline(0, color="red", ls="--", lw=1)
axes[0].axhline(np.pi, color="gray", ls=":", lw=0.8)
axes[0].axhline(-np.pi, color="gray", ls=":", lw=0.8)
axes[0].set_xlabel("시간 (s)"); axes[0].set_ylabel("각도 $\\theta$ (rad, 0 = 위쪽 수직)")
axes[0].set_title("각도 $\\theta(t)$"); axes[0].legend(fontsize=9)
axes[1].plot(tt, td1, color="green", label="PD ($k_p$=8, $k_d$=2)  <- 최선")
axes[1].plot(tt, td2, color="orange", label="PD ($k_p$=4, $k_d$=1)")
axes[1].axhline(0, color="red", ls="--", lw=1)
axes[1].set_xlabel("시간 (s)"); axes[1].set_ylabel("각속도 $\\dot\\theta$ (rad/s)")
axes[1].set_title("각속도 $\\dot\\theta(t)$"); axes[1].legend(fontsize=9)
fig.tight_layout()
fig.savefig(IMG + "/ch13_2_pd_trajectory.svg", bbox_inches="tight")
plt.show()

print(f"(8,2) 마지막: theta={th1[-1]:+.3f} rad, thdot={td1[-1]:+.3f} rad/s  ->  위쪽 균형에 수렴(유지)")
print(f"(4,1) 마지막: theta={th2[-1]:+.3f} rad, thdot={td2[-1]:+.3f} rad/s  ->  여전히 큰 진동(수렴하지 않음)")

(8,2) 마지막: theta=+0.000 rad, thdot=-0.000 rad/s  ->  위쪽 균형에 수렴(유지)
(4,1) 마지막: theta=-1.637 rad, thdot=-3.049 rad/s  ->  여전히 큰 진동(수렴하지 않음)


## 6. 로봇팔·사족보행: Reacher-v5 vs Ant-v5

여기서부터는 실제 MuJoCo 환경을 사용한다 — 상태·행동공간의 크기와 무작위 정책의 점수를, **시드 0~2로** 재본다(단일 시드는 운이다).

In [6]:
def run_random_ep(name, seed, n_steps=100):
    """에피소드마다 시드된 RNG로 무작위 행동을 뽑는다 (재현 가능)."""
    env = gym.make(name)
    lo, hi = env.action_space.low, env.action_space.high
    rng = np.random.default_rng(1000 + seed)
    s, _ = env.reset(seed=seed)
    tot, length, end_type = 0.0, 0, None
    for t in range(n_steps):
        s, r, term, trunc, info = env.step(rng.uniform(lo, hi))
        tot += r
        length = t + 1
        if term or trunc:
            end_type = "terminated" if term else "truncated"
            break
    if end_type is None:
        end_type = f"{n_steps}스텝 완주"
    env.close()
    return tot, length, end_type

for name in ["Reacher-v5", "Ant-v5"]:
    env = gym.make(name)
    print(name, "상태공간:", env.observation_space.shape,
          "행동공간:", env.action_space.shape, env.action_space.low[:2], env.action_space.high[:2])
    for seed in range(3):
        tot, length, end_type = run_random_ep(name, seed)
        print(f"  seed{seed}: 리턴 {tot:8.2f}  (길이 {length:3d}스텝, {end_type})")
    s, _ = env.reset(seed=0)
    s, r, term, trunc, info = env.step(env.action_space.sample())
    print("  info keys:", sorted(info.keys()))
    env.close()
    print()

Reacher-v5 상태공간: (10,) 행동공간: (2,) [-1. -1.] [1. 1.]
  seed0: 리턴   -42.98  (길이  50스텝, truncated)
  seed1: 리턴   -40.02  (길이  50스텝, truncated)
  seed2: 리턴   -39.75  (길이  50스텝, truncated)
  info keys: ['reward_ctrl', 'reward_dist']

Ant-v5 상태공간: (105,) 행동공간: (8,) [-1. -1.] [1. 1.]
  seed0: 리턴    -6.88  (길이  63스텝, terminated)
  seed1: 리턴   -19.47  (길이  24스텝, terminated)
  seed2: 리턴    -9.94  (길이  11스텝, terminated)
  info keys: ['distance_from_origin', 'reward_contact', 'reward_ctrl', 'reward_forward', 'reward_survive', 'x_position', 'x_velocity', 'y_position', 'y_velocity']



## 7. Ant의 4항 장부: "+3.60" 에피소드 해부

무작위 Ant가 **양수** 리턴을 낸 에피소드(seed 4)는 16스텝 만에 `terminated`(넘어짐)으로 끝났다 — 4항 보상을 스텝마다 따로 모아, 리턴이 어디서 와서 어디로 사라졌는지 재구성한다(에피소드별 시드된 RNG라 언제 다시 돌려도 같은 숫자):

$$r = \underbrace{1.0 \times \mathbb{1}[\text{healthy}]}_{\text{생존}} + \underbrace{1.0 \times \dot{x}}_{\text{전진}} - \underbrace{0.5\,\lVert a \rVert^2}_{\text{제어}} - \underbrace{5\times10^{-4}\,\sum F_{\text{contact}}^2}_{\text{접촉}}$$

In [7]:
env = gym.make("Ant-v5")
lo, hi = env.action_space.low, env.action_space.high
rng = np.random.default_rng(1004)  # seed4 에피소드(양수 리턴, 16스텝에 terminated) — 재현 가능
s, _ = env.reset(seed=4)
x0 = s[0]
surv = fwd = ctrl = cont = tot = 0.0
length = 0
for t in range(100):
    s, r, term, trunc, info = env.step(rng.uniform(lo, hi))
    tot += r
    surv += info["reward_survive"]
    fwd += info["reward_forward"]
    ctrl += info["reward_ctrl"]
    cont += info["reward_contact"]
    length = t + 1
    if term or trunc:
        break
print(f"seed4 에피소드: 길이 {length}스텝, terminated={term}, 전진 거리 {s[0] - x0:+.2f} m")
print(f"  생존   : {surv:+8.2f}  (healthy 스텝당 1.0)")
print(f"  전진   : {fwd:+8.2f}  (1.0 × x-방향 속도)")
print(f"  제어   : {ctrl:+8.2f}  (0.5 × ||a||^2)")
print(f"  접촉   : {cont:+8.2f}  (5e-4 × 접촉력 합)")
print(f"  총계   : {surv + fwd + ctrl + cont:+8.2f}  (실제 리턴 {tot:+.2f}와 일치해야 함)")
print("\n→ '양수 리턴' ≠ '잘 걷는다': 15스텝의 생존점+전진이 제어비용을 비로소 상회한, 16스텝 만에 넘어진 에피소드다")
env.close()

seed4 에피소드: 길이 16스텝, terminated=True, 전진 거리 +0.16 m
  생존   :   +15.00  (healthy 스텝당 1.0)
  전진   :   +10.51  (1.0 × x-방향 속도)
  제어   :   -21.90  (0.5 × ||a||^2)
  접촉   :    -0.01  (5e-4 × 접촉력 합)
  총계   :    +3.60  (실제 리턴 +3.60와 일치해야 함)

→ '양수 리턴' ≠ '잘 걷는다': 15스텝의 생존점+전진이 제어비용을 비로소 상회한, 16스텝 만에 넘어진 에피소드다


## 8. 스텝당 시간: "큰 로봇"의 계산 대가

물리 시뮬레이션이 무거워지는 것을 측정한다(2000스텝, 무작위 행동). Ant가 Reacher의 약 5배지만 둘 다 CPU에서 학습 가능한 속도다 — 실무의 병목은 스텝당 시간이 아니라 *필요 스텝 수*다(13.3절 PPO는 Reacher에서 100만 스텝을 쓴다).

In [8]:
import time
for name in ["Pendulum-v1", "Reacher-v5", "Ant-v5"]:
    env = gym.make(name)
    s, _ = env.reset(seed=0)
    n = 2000
    t0 = time.time()
    for _ in range(n):
        s, r, term, trunc, info = env.step(env.action_space.sample())
        if term or trunc:
            s, _ = env.reset()
    dt = (time.time() - t0) / n
    print(f"  {name:<13}: {dt * 1e6:7.1f} µs/스텝  ({n * dt:5.2f}s / 2000스텝)")
    env.close()

  Pendulum-v1  :    24.8 µs/스텝  ( 0.05s / 2000스텝)


  Reacher-v5   :    39.0 µs/스텝  ( 0.08s / 2000스텝)


  Ant-v5       :   184.4 µs/스텝  ( 0.37s / 2000스텝)


## 정리

1. **평가 프로토콜이 먼저다** — 50스텝 창에서는 무작위와 PD를 구별 못 하지만, 창을 200스텝으로 *양쪽에 같이* 늘리면 PD(8,2)가 무작위 대비 누적 벌을 절반 이하로 줄인다. 비교할 때 창·시드·초기 분포가 같아야 차이는 "정책의 차이"가 된다.
2. **그리드는 고차원에서 죽는다** — PD는 $(k_p, k_d)$ 두 숫자의 20개 조합을 훑는 것으론 $\approx -455$까지 갔지만, Ant의 정책은 105→8차원 함수다. "데이터로부터 정책을 배운다"는 것이 13.3절 PPO의 동기다.
3. **점수는 프록시다** — 최선 PD(8,1)가 진자를 *세워 유지*하는지 궤적으로 확인해야 하고($k_p < 5$인 (4,1)은 10초 내내 진동), Ant의 "양수 리턴"은 15스텝 만에 넘어진 에피소드일 수 있다 — 리턴 평균과 함께 에피소드 길이·종료 비율을 반드시 함께 보고한다.

다음(13.3절)에서는 이 Reacher 환경에 Chapter 11의 PPO를 적용해, **신경망이 20개 PD 중의 하나도 아닌 새로운 정책을 *배우는* 과정**을 추적한다.